In [4]:
"""
FAULT LINE REPAIRABILITY ANALYSIS
==================================
Additions to existing paired spares analysis code for radial fault lines
"""
from repair_analysis_final import build_chain_with_paired_spares, simulate_repair_with_paired_spares
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# ============================================================================
# FAULT LINE GENERATION
# ============================================================================

def generate_radial_fault_line(N, angle_degrees, resolution):
    """
    Generate a fault line passing through grid center at given angle.
    
    Parameters:
    - N: grid size
    - angle_degrees: angle from 0-180° (0°=horizontal, 90°=vertical)
    - resolution: number of points to sample along line (default 50 for clean lines)
    
    Returns:
    - fault_positions: set of (i, j) positions along the line
    """
    center_x = (N - 1) / 2.0
    center_y = (N - 1) / 2.0
    
    angle_rad = np.radians(angle_degrees)
    fault_positions = set()
    
    # Extend line far enough to cross entire grid
    max_dist = N * 1.5
    
    for k in range(resolution + 1):
        t = -max_dist + (2 * max_dist * k) / resolution
        x = center_x + t * np.cos(angle_rad)
        y = center_y + t * np.sin(angle_rad)
        
        # Round to nearest grid position
        i = int(round(y))
        j = int(round(x))
        
        if 0 <= i < N and 0 <= j < N:
            fault_positions.add((i, j))
    
    return fault_positions


def generate_all_radial_lines(N, angle_step, resolution):
    """
    Generate all radial fault lines at specified angle intervals.
    
    Parameters:
    - N: grid size
    - angle_step: angle increment in degrees (e.g., 1, 5, 10, 15)
    - resolution: sampling resolution for each line
    
    Returns:
    - lines_dict: {angle: set of (i,j) positions}
    """
    angles = range(0, 180, angle_step)
    lines_dict = {}
    
    for angle in angles:
        fault_positions = generate_radial_fault_line(N, angle, resolution)
        lines_dict[angle] = fault_positions
    
    return lines_dict


# ============================================================================
# VISUALIZATION OF ALL FAULT LINES
# ============================================================================

def plot_all_fault_lines(N, lines_dict, title="All Radial Fault Lines"):
    """
    Plot all generated fault lines with unique colors.
    
    Parameters:
    - N: grid size
    - lines_dict: {angle: set of fault positions}
    """
    fig, ax = plt.subplots(figsize=(14, 14))
    
    # Create a color map for different angles
    num_lines = len(lines_dict)
    colors = plt.cm.hsv(np.linspace(0, 0.95, num_lines))
    
    # Create grid to track which line(s) hit each bump
    grid_colors = np.full((N, N), -1, dtype=int)  # -1 = no line
    
    # Assign each bump to a line (first line to hit it gets the color)
    for line_idx, (angle, fault_positions) in enumerate(sorted(lines_dict.items())):
        for (i, j) in fault_positions:
            if grid_colors[i, j] == -1:
                grid_colors[i, j] = line_idx
    
    # Plot each bump with its assigned color
    for i in range(N):
        for j in range(N):
            if grid_colors[i, j] >= 0:
                color_idx = grid_colors[i, j]
                ax.plot(j, N - 1 - i, 'o', color=colors[color_idx], 
                       markersize=8, markeredgecolor='black', markeredgewidth=0.5)
            else:
                ax.plot(j, N - 1 - i, 'o', color='lightgray', 
                       markersize=6, alpha=0.3)
    
    # Add grid center
    center = (N - 1) / 2.0
    ax.plot(center, N - 1 - center, '*', color='red', 
           markersize=20, markeredgecolor='darkred', markeredgewidth=2,
           label='Grid Center')
    
    ax.set_xlim(-1, N)
    ax.set_ylim(-1, N)
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_title(f"{title}\n{num_lines} lines at {sorted(lines_dict.keys())[0]}° to {sorted(lines_dict.keys())[-1]}° intervals", 
                fontsize=14)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()


# ============================================================================
# REPAIRABILITY ANALYSIS FOR FAULT LINES
# ============================================================================

def run_fault_line_repair_sweep(color_grid, K, redundancy_ratio, 
                                angle_step, resolution, 
                                plot_all_lines):
    """
    Run repairability analysis for radial fault lines.
    
    Parameters:
    - color_grid: NxN grid with chain assignments
    - K: number of chains
    - redundancy_ratio: spare redundancy ratio
    - angle_step: angle increment in degrees (default 10°)
    - resolution: line sampling resolution (default 50)
    - plot_all_lines: if True, show visualization of all fault lines
    
    Returns:
    - summary: dictionary with overall statistics
    """
    N = color_grid.shape[0]
    
    # Build chains with paired spares (using existing function)
    chain_data = build_chain_with_paired_spares(color_grid, K, redundancy_ratio)
    
    # Generate all fault lines
    lines_dict = generate_all_radial_lines(N, angle_step, resolution)
    
    # Optional: visualize all lines
    if plot_all_lines:
        plot_all_fault_lines(N, lines_dict, 
                           title=f"Fault Lines at {angle_step}° Intervals")
    
    # Run repair simulation for each line
    total_faults = 0
    total_repaired = 0
    total_ignored_spares = 0
    total_unrepairable = 0
    perfect_lines = 0
    
    results = []
    
    print("\n" + "=" * 90)
    print(f"FAULT LINE REPAIRABILITY SWEEP (Angle Step: {angle_step}°, Resolution: {resolution})")
    print("=" * 90)
    print(f"{'Line#':>6} | {'Angle':>6} | {'Faults':>7} | {'Repaired':>9} | "
          f"{'Ignored':>8} | {'Unrepair.':>10} | {'Rate(%)':>9} | {'Perfect':>8}")
    print("-" * 90)
    
    for line_idx, (angle, fault_positions) in enumerate(sorted(lines_dict.items()), 1):
        # Run repair simulation (using existing function)
        stats, repair_assignments = simulate_repair_with_paired_spares(
            color_grid, chain_data, fault_positions
        )
        
        repaired = stats["repaired"]
        total = stats["total_faults"]
        ignored = stats.get("ignored_spare_faults", 0)
        unrepairable = stats["unrepairable"]
        rate = (100 * repaired / total) if total > 0 else 0.0
        
        # Check if perfect repair (100%)
        is_perfect = (total > 0 and repaired == total)
        if is_perfect:
            perfect_lines += 1
        
        # Accumulate totals
        total_faults += total
        total_repaired += repaired
        total_ignored_spares += ignored
        total_unrepairable += unrepairable
        
        # Print line result
        perfect_str = "YES" if is_perfect else ""
        print(f"{line_idx:>6} | {angle:>6}° | {total:>7} | {repaired:>9} | "
              f"{ignored:>8} | {unrepairable:>10} | {rate:>9.2f} | {perfect_str:>8}")
        
        results.append({
            "line_id": line_idx,
            "angle": angle,
            "total_faults": total,
            "repaired": repaired,
            "ignored_spares": ignored,
            "unrepairable": unrepairable,
            "repair_rate": rate,
            "is_perfect": is_perfect
        })
    
    # Summary statistics
    print("=" * 90)
    print("SUMMARY")
    print("=" * 90)
    
    num_lines = len(lines_dict)
    perfect_percentage = (100 * perfect_lines / num_lines) if num_lines > 0 else 0.0
    
    print(f"Total lines tested: {num_lines}")
    print(f"Perfect repairs (100%): {perfect_lines} / {num_lines} ({perfect_percentage:.2f}%)")
    
    if total_faults > 0:
        overall_rate = 100 * total_repaired / total_faults
        print(f"\nTotal signal faults across all lines: {total_faults}")
        print(f"Total repaired: {total_repaired}")
        print(f"Total unrepairable: {total_unrepairable}")
        print(f"Total ignored (on spares): {total_ignored_spares}")
        print(f"Overall repair rate: {overall_rate:.2f}%")
    else:
        overall_rate = 0.0
        print("\nNo valid signal faults in any line.")
    
    print("=" * 90)
    
    return {
        "results": results,
        "num_lines": num_lines,
        "perfect_lines": perfect_lines,
        "perfect_percentage": perfect_percentage,
        "total_faults": total_faults,
        "total_repaired": total_repaired,
        "total_unrepairable": total_unrepairable,
        "total_ignored_spares": total_ignored_spares,
        "overall_repair_rate": overall_rate
    }


# ============================================================================
# PARAMETER SWEEP FOR MULTIPLE CONFIGURATIONS
# ============================================================================

def run_fault_line_parameter_sweep(color_grid, K, 
                                   redundancy_ratios,
                                   angle_steps,
                                   resolution,
                                   plot_lines_for_first):
    """
    Run fault line repairability sweep across multiple parameter combinations.
    
    Parameters:
    - color_grid: NxN grid
    - K: number of chains
    - redundancy_ratios: list of redundancy ratios to test
    - angle_steps: list of angle step sizes to test
    - resolution: line sampling resolution
    - plot_lines_for_first: if True, plot lines for first configuration
    """
    summary_table = []
    
    for rr_idx, rr in enumerate(redundancy_ratios):
        for step_idx, angle_step in enumerate(angle_steps):
            print(f"\n{'#'*90}")
            print(f"CONFIGURATION: Redundancy 1:{rr}, Angle Step {angle_step}°")
            print(f"{'#'*90}")
            
            # Plot lines only for first configuration
            plot_flag = plot_lines_for_first and (rr_idx == 0) and (step_idx == 0)
            
            summary = run_fault_line_repair_sweep(
                color_grid, K, rr, 
                angle_step=angle_step,
                resolution=resolution,
                plot_all_lines=plot_flag
            )
            
            summary_table.append({
                "redundancy_ratio": rr,
                "angle_step": angle_step,
                "num_lines": summary["num_lines"],
                "perfect_lines": summary["perfect_lines"],
                "perfect_percentage": summary["perfect_percentage"],
                "total_faults": summary["total_faults"],
                "repaired": summary["total_repaired"],
                "unrepairable": summary["total_unrepairable"],
                "ignored": summary["total_ignored_spares"],
                "repair_rate": summary["overall_repair_rate"]
            })
    
    # Print summary table
    print("\n" + "=" * 110)
    print("OVERALL SUMMARY TABLE — FAULT LINE REPAIRABILITY")
    print("=" * 110)
    print(f"{'Redundancy':>12} | {'AngleStep':>10} | {'#Lines':>7} | "
          f"{'Perfect':>8} | {'Yield%':>7} | {'Faults':>8} | {'Repaired':>9} | "
          f"{'Unrepair':>9} | {'Ignored':>8} | {'Rate%':>7}")
    print("-" * 110)
    
    for row in summary_table:
        print(f"{row['redundancy_ratio']:>12} | "
              f"{row['angle_step']:>10}° | "
              f"{row['num_lines']:>7} | "
              f"{row['perfect_lines']:>8} | "
              f"{row['perfect_percentage']:>7.2f} | "
              f"{row['total_faults']:>8} | "
              f"{row['repaired']:>9} | "
              f"{row['unrepairable']:>9} | "
              f"{row['ignored']:>8} | "
              f"{row['repair_rate']:>7.2f}")
    
    print("=" * 110)
    
    return summary_table


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    import pickle
    
    # Load your optimized grid
    with open('optimized_grid_NKM_20-7-5.pkl', 'rb') as f:
        data = pickle.load(f)
    
    optimized_grid = data['grid']
    K = data['K']
    N = data['N']
    res = 75
    print("=" * 70)
    print("FAULT LINE REPAIRABILITY ANALYSIS")
    print("=" * 70)
    print(f"Grid: {N}x{N}, {K} chains")
    print("=" * 70)
    
    # Example 1: Single configuration with visualization
    # print("\n--- EXAMPLE 1: Single configuration with line visualization ---")
    # summary = run_fault_line_repair_sweep(
    #     optimized_grid, 
    #     K, 
    #     redundancy_ratio=8,
    #     angle_step=15,
    #     resolution=res,
    #     plot_all_lines=True
    # )
    
    # Example 2: Parameter sweep
    print("\n--- EXAMPLE 2: Parameter sweep ---")
    summary_table = run_fault_line_parameter_sweep(
        optimized_grid,
        K,
        redundancy_ratios=[16],
        angle_steps=[1],
        resolution=res,
        plot_lines_for_first=False
    )

FAULT LINE REPAIRABILITY ANALYSIS
Grid: 20x20, 7 chains

--- EXAMPLE 2: Parameter sweep ---

##########################################################################################
CONFIGURATION: Redundancy 1:16, Angle Step 1°
##########################################################################################

FAULT LINE REPAIRABILITY SWEEP (Angle Step: 1°, Resolution: 75)
 Line# |  Angle |  Faults |  Repaired |  Ignored |  Unrepair. |   Rate(%) |  Perfect
------------------------------------------------------------------------------------------
     1 |      0° |      17 |        16 |        3 |          1 |     94.12 |         
     2 |      1° |      18 |        17 |        2 |          1 |     94.44 |         
     3 |      2° |      18 |        17 |        2 |          1 |     94.44 |         
     4 |      3° |      18 |        17 |        2 |          1 |     94.44 |         
     5 |      4° |      18 |        17 |        2 |          1 |     94.44 |         
     6 |

In [5]:
import matplotlib.pyplot as plt
import numpy as np

# ======================================================
# FONT / SIZE PARAMETERS (MATCH YOUR PREVIOUS STYLE)
# ======================================================
fontsize_title = 18
fontsize_label = 18
fontsize_tick = 20
bar_width = 0.25

# ======================================================
# EXTRACT VALUES FROM summary_table
# ======================================================
redundancy_vals = sorted(
    list(set(row["redundancy_ratio"] for row in summary_table)),
    reverse=True
)

repair_rates = []
ignored_vals = []

for rr in redundancy_vals:
    for row in summary_table:
        if row["redundancy_ratio"] == rr:
            repair_rates.append(row["repair_rate"])
            ignored_vals.append(row["ignored"])

repair_rates = np.array(repair_rates)
ignored_vals = np.array(ignored_vals)

# ======================================================
# IMPROVEMENT (1:8 vs 1:16, 1:4 vs 1:16)
# ======================================================
def get_rr_rate(rr):
    for row in summary_table:
        if row["redundancy_ratio"] == rr:
            return row["repair_rate"]
    return None

r16 = get_rr_rate(16)
r8  = get_rr_rate(8)
r4  = get_rr_rate(4)

improvement = np.array([
    ((r8 - r16) / r16) * 100,
    ((r4 - r16) / r16) * 100
])

# ======================================================
# ==========   PLOT 3 SUBPLOTS   =======================
# ======================================================

fig, axes = plt.subplots(3, 1, figsize=(10, 8))

# ------------------------------------------------------
# SUBPLOT 1 — Repairability
# ------------------------------------------------------
ax = axes[0]
x = np.arange(len(redundancy_vals))

ax.bar(x, repair_rates, width=bar_width)

ax.set_xticks(x)
ax.set_xticklabels([f"1:{rr}" for rr in redundancy_vals], fontsize=fontsize_tick)
ax.set_ylabel("Repairability (%)", fontsize=fontsize_label)
#ax.set_title("Fault-Line Repairability vs Redundancy Ratio", fontsize=fontsize_title)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.tick_params(axis='y', labelsize=fontsize_tick)

# ------------------------------------------------------
# SUBPLOT 2 — Ignored Spare Faults
# ------------------------------------------------------
ax = axes[1]

ax.bar(x, ignored_vals, width=bar_width)

ax.set_xticks(x)
ax.set_xticklabels([f"1:{rr}" for rr in redundancy_vals], fontsize=fontsize_tick)
ax.set_ylabel("Spare Faults", fontsize=fontsize_label)
#ax.set_title("Line-defects Spare Faults vs Redundancy Ratio", fontsize=fontsize_title)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.tick_params(axis='y', labelsize=fontsize_tick)

# ------------------------------------------------------
# SUBPLOT 3 — Improvement
# ------------------------------------------------------
ax = axes[2]
imp_x = np.arange(2)
imp_labels = ["1:8 vs 1:16", "1:4 vs 1:16"]

ax.bar(imp_x, improvement, width=bar_width)

ax.set_xticks(imp_x)
ax.set_xticklabels(imp_labels, fontsize=fontsize_tick)
ax.set_ylabel("Improvement (%)", fontsize=fontsize_label)
#ax.set_title("Repairability Improvement from 1:16", fontsize=fontsize_title)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.tick_params(axis='y', labelsize=fontsize_tick)

plt.tight_layout()
plt.show()


TypeError: unsupported operand type(s) for -: 'NoneType' and 'float'